# 05. Xây Dựng và Đối Chuẩn Mô Hình Baseline với 5-Fold Stratified Group Cross-Validation
## Giai đoạn 7A (Tuần 7): 5 Mô hình Baseline, So sánh Single-Split vs 5-Fold CV, Ma trận Nhầm lẫn OOF & Phân tích Lỗi
**Dự án**: LMS Tiên tiến và Learning Analytics (Bộ dữ liệu OULAD)  
**Tác giả**: Vũ Hoà Bình (MSV: 202416137)  

---

### Mục tiêu & Quy chuẩn Thực nghiệm (Zero Data Leakage, Tránh Ăn May):
1. **Chiến lược 5-Fold Stratified Group Cross-Validation (`StratifiedGroupKFold(n_splits=5)`)**:
   - Chia toàn bộ dữ liệu thành **5 phần đều nhau (~20% mỗi phần)** phân tầng theo biến mục tiêu `at_risk` và nhóm theo `id_student` ($0$ sinh viên trùng lặp giữa Train và Test ở bất kỳ Fold nào).
   - Lần lượt lấy phần $1 ightarrow 5$ làm **Test ($20\%$)** và 4 phần còn lại làm **Train ($80\%$)**, đảm bảo $100\%$ sinh viên đều được đánh giá độc lập đúng 1 lần (**Out-of-Fold / OOF**).
2. **Tiền xử lý In-Fold Zero Data Leakage**:
   - Toàn bộ tham số Imputation (`imd_band` Mode, `date_registration` Median), chặn ngoại lai ($Q_3 + 3 	imes 	ext{IQR}$), biến đổi `log1p` (bảo toàn dấu âm của `clicks_trend`), `OneHotEncoder` và `RobustScaler` đều **chỉ fit trên 4 phần Train** của từng Fold rồi `.transform()` sang phần Test tương ứng.
3. **5 Mô hình Baseline theo Đề cương**:
   - `Logistic Regression`, `Decision Tree`, `Random Forest`, `SVM` (`LinearSVC` hiệu chuẩn xác suất `CalibratedClassifierCV`), `KNN`.
4. **Bộ Chỉ số Toàn diện & Lựa chọn Top Baseline**:
   - Đo lường 6 chỉ số (`Accuracy`, `Precision`, `Recall`, `F1-Score`, `AUC-ROC`, `AUC-PR`) + `Brier Score` cho từng Fold $1 ightarrow 5$, tổng hợp $	ext{Mean} \pm 	ext{SD}$ và OOF, ưu tiên **Recall** cho bài toán cảnh báo sớm.
   - Thực hiện phân tích lỗi (Error Analysis) trên dự đoán OOF và chọn Top 3 mô hình Baseline chuyển sang Giai đoạn 7B (`06_advanced_models.ipynb`).

In [1]:
import json
import logging
import sys
import time
import unittest
import warnings
from pathlib import Path
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

if sys.platform == "win32":
    try:
        sys.stdout.reconfigure(encoding="utf-8")
        sys.stderr.reconfigure(encoding="utf-8")
    except Exception:
        pass

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
logger = logging.getLogger("OULAD_Week7_Baselines")

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: "%.4f" % x)

PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SNAPSHOTS_DIR = PROCESSED_DIR / "week5_6_snapshots"
WEEK7_PROCESSED_DIR = PROCESSED_DIR / "week7_modeling"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = PROJECT_ROOT / "models"

for d in [PROCESSED_DIR, SNAPSHOTS_DIR, WEEK7_PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project Root : {PROJECT_ROOT}")
print(f"Processed Dir: {PROCESSED_DIR}")
print(f"Figures Dir  : {FIGURES_DIR}")

Project Root : C:\Users\mrore\Project_I
Processed Dir: C:\Users\mrore\Project_I\data\processed
Figures Dir  : C:\Users\mrore\Project_I\reports\figures


## 1. Động cơ Kiểm chứng Chéo `OULAD5FoldCVEngine` (5-Fold Stratified Group CV & In-Fold Zero Leakage)
Cài đặt lớp tự chủ hoàn toàn để nạp các bảng snapshot (`25%`, `50%`, `75%`, `100%`), lọc Cohort (`full`, `pre_course_active`, `dynamic_active`), chia 5 phần đều nhau theo `id_student` và chuẩn hóa không gian 114 biến (`week6_advanced`) bên trong từng vòng lặp Fold.

In [2]:
class OULAD5FoldCVEngine:
    """
    Động cơ chia 5 phần StratifiedGroupKFold theo id_student và tiền xử lý In-Fold Zero Data Leakage
    cho hệ 114 đặc trưng mã hóa (week6_advanced).
    """

    ALL_ACTIVITY_TYPES = [
        "dataplus", "dualpane", "externalquiz", "folder", "forumng",
        "glossary", "homepage", "htmlactivity", "oucollaborate", "oucontent",
        "ouelluminate", "ouwiki", "page", "questionnaire", "quiz",
        "repeatactivity", "resource", "sharedsubpage", "subpage", "url",
    ]

    ORDINAL_MAPPINGS = {
        "highest_education": {
            "No Formal quals": 0,
            "Lower Than A Level": 1,
            "A Level or Equivalent": 2,
            "HE Qualification": 3,
            "Post Graduate Qualification": 4,
        },
        "imd_band": {
            "0-10%": 0, "10-20": 1, "20-30%": 2, "30-40%": 3, "40-50%": 4,
            "50-60%": 5, "60-70%": 6, "70-80%": 7, "80-90%": 8, "90-100%": 9,
            "Unknown": 4,
        },
        "age_band": {"0-35": 0, "35-55": 1, "55<=": 2},
    }

    NOMINAL_COLS = ["gender", "disability", "region", "code_module"]
    ORDINAL_COLS = ["highest_education", "imd_band", "age_band"]
    FLAG_COLS = ["imd_was_missing", "has_no_submissions", "has_no_clicks"]

    @classmethod
    def get_week6_numeric_cols(cls) -> List[str]:
        all_20_act_cols = [f"clicks_{act}" for act in cls.ALL_ACTIVITY_TYPES]
        return [
            # 5.1 Click-based (11)
            "total_clicks", "active_days", "clicks_per_day", "clicks_per_active_day",
            "click_std", "click_cv", "active_days_in_course", "active_days_ratio",
            "pre_course_clicks", "has_pre_course_clicks", "active_span_days",
            # 5.2 Activity-based (31)
            *all_20_act_cols,
            "cognitive_clicks", "social_clicks", "oucontent_ratio", "forumng_ratio",
            "quiz_ratio", "resource_ratio", "glossary_ratio", "cognitive_ratio",
            "social_ratio", "activity_diversity", "activity_entropy",
            # 5.3 Assessment-based (19)
            "total_course_assessments", "due_course_assessments", "avg_score",
            "weighted_avg_score", "earned_weight_score", "min_score", "max_score",
            "score_std", "first_score", "num_submissions", "num_submissions_raw",
            "submission_rate", "course_progress_submission_rate", "avg_days_before_deadline",
            "late_submissions", "late_submission_rate", "tma1_score", "tma1_days_early",
            "submitted_tma1",
            # 5.4 Demographics & Context numeric (7)
            "num_of_prev_attempts", "studied_credits", "module_presentation_length",
            "days_before_start", "is_repeat_student", "credit_overload", "is_late_registration",
            # 6.1 Temporal & Decay (8)
            "clicks_week_1_to_4", "clicks_last_2_weeks", "clicks_trend", "engagement_decay",
            "days_since_last_activity", "inactivity_ratio", "max_inactivity_streak", "consistency_score",
            # 6.2 Regularity, Weekend & Burst (6)
            "active_weeks", "login_regularity", "weekend_clicks", "weekend_ratio",
            "burst_count", "burst_ratio",
            # 6.3 Score Trajectory (2)
            "last_minus_first_score", "score_trend_slope",
        ]

    @staticmethod
    def filter_cohort(df: pd.DataFrame, cohort_type: str = "dynamic_active") -> pd.DataFrame:
        if cohort_type == "full":
            return df.copy().reset_index(drop=True)
        elif cohort_type == "pre_course_active":
            mask = df["date_unregistration"].isna() | (df["date_unregistration"] >= 0)
            return df[mask].copy().reset_index(drop=True)
        elif cohort_type == "dynamic_active":
            mask = df["date_unregistration"].isna() | (df["date_unregistration"] > df["cutoff_day"])
            return df[mask].copy().reset_index(drop=True)
        raise ValueError(f"cohort_type không hợp lệ: {cohort_type}")

    @staticmethod
    def get_5fold_splits(
        df: pd.DataFrame, n_splits: int = 5, random_state: int = 42
    ) -> Tuple[List[Tuple[int, np.ndarray, np.ndarray]], pd.DataFrame]:
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        y = df["at_risk"].values
        groups = df["id_student"].values
        splits = []
        audit_rows = []
        total_n = len(df)
        for fold, (tr_idx, te_idx) in enumerate(sgkf.split(df, y, groups=groups), start=1):
            tr_students = set(groups[tr_idx])
            te_students = set(groups[te_idx])
            overlap = len(tr_students.intersection(te_students))
            splits.append((fold, tr_idx, te_idx))
            audit_rows.append({
                "fold": fold,
                "n_train": int(len(tr_idx)),
                "n_test": int(len(te_idx)),
                "pct_test": round(100.0 * len(te_idx) / total_n, 2),
                "n_train_students": int(len(tr_students)),
                "n_test_students": int(len(te_students)),
                "train_at_risk_rate": round(float(y[tr_idx].mean()), 4),
                "test_at_risk_rate": round(float(y[te_idx].mean()), 4),
                "student_overlap": int(overlap),
            })
        return splits, pd.DataFrame(audit_rows)

    def prepare_fold_matrices(
        self, train_df: pd.DataFrame, test_df: pd.DataFrame
    ) -> Tuple[pd.DataFrame, pd.DataFrame]:
        num_cols = self.get_week6_numeric_cols()

        imd_mode = (
            train_df["imd_band"].mode().iloc[0]
            if not train_df["imd_band"].mode().empty
            else "20-30%"
        )
        reg_median = float(train_df["date_registration"].median())

        skewed_cols = [
            c for c in num_cols
            if (
                (c.startswith("clicks_") and c != "clicks_trend")
                or c in [
                    "total_clicks", "active_days", "active_days_in_course",
                    "daily_click_mean", "daily_click_std", "click_std",
                    "pre_course_clicks", "cognitive_clicks", "social_clicks",
                    "weekend_clicks", "studied_credits",
                ]
            )
        ]

        bounds: Dict[str, Tuple[float, float]] = {}
        for col in skewed_cols:
            s = train_df[col].dropna()
            q25, q75 = float(s.quantile(0.25)), float(s.quantile(0.75))
            iqr = q75 - q25
            if iqr > 0:
                bounds[col] = (0.0, q75 + 3.0 * iqr)

        ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        ohe.fit(train_df[self.NOMINAL_COLS].fillna("Missing").astype(str))
        ohe_names = list(ohe.get_feature_names_out(self.NOMINAL_COLS))

        def _clean_subset(df_in: pd.DataFrame) -> pd.DataFrame:
            d = df_in.copy()
            d["imd_was_missing"] = d["imd_band"].isna().astype(float)
            d["has_no_submissions"] = (d["num_submissions"].fillna(0) == 0).astype(float)
            d["has_no_clicks"] = (d["total_clicks"].fillna(0) == 0).astype(float)

            d["imd_band"] = d["imd_band"].fillna(imd_mode)
            d["date_registration"] = d["date_registration"].fillna(reg_median)
            d["days_before_start"] = -d["date_registration"]

            score_cols = [
                "avg_score", "weighted_avg_score", "earned_weight_score",
                "min_score", "max_score", "score_std", "first_score", "tma1_score",
            ]
            for sc in score_cols:
                if sc in d.columns:
                    d[sc] = d[sc].fillna(0.0)

            early_cols = ["avg_days_early", "avg_days_before_deadline", "tma1_days_early"]
            for ec in early_cols:
                if ec in d.columns:
                    d[ec] = d[ec].fillna(-10.0)

            for c in num_cols:
                if c in d.columns:
                    d[c] = d[c].fillna(0.0)

            for col, mapping in self.ORDINAL_MAPPINGS.items():
                d[col] = d[col].map(mapping).fillna(mapping.get("Unknown", 2)).astype(float)

            for col, (low, high) in bounds.items():
                if col in d.columns:
                    d[col] = d[col].clip(lower=low, upper=high)

            for col in skewed_cols:
                if col in d.columns and "click" in col and "ratio" not in col and "per" not in col and "cv" not in col:
                    d[col] = np.log1p(np.maximum(0.0, d[col]))

            return d

        tr_clean = _clean_subset(train_df)
        te_clean = _clean_subset(test_df)

        cols_to_scale = num_cols + self.ORDINAL_COLS
        scaler = RobustScaler()
        scaler.fit(tr_clean[cols_to_scale])

        def _assemble(df_clean: pd.DataFrame, df_orig: pd.DataFrame) -> pd.DataFrame:
            scaled_arr = scaler.transform(df_clean[cols_to_scale])
            scaled_df = pd.DataFrame(scaled_arr, columns=cols_to_scale, index=df_orig.index)
            ohe_arr = ohe.transform(df_orig[self.NOMINAL_COLS].fillna("Missing").astype(str))
            ohe_df = pd.DataFrame(ohe_arr, columns=ohe_names, index=df_orig.index)
            flags_df = df_clean[self.FLAG_COLS].copy()
            out = pd.concat([scaled_df, ohe_df, flags_df], axis=1)
            return out.replace([np.inf, -np.inf], 0.0).fillna(0.0)

        return _assemble(tr_clean, train_df), _assemble(te_clean, test_df)

    @staticmethod
    def evaluate_predictions(
        y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.50
    ) -> Dict[str, Any]:
        y_pred = (y_prob >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        return {
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "recall": float(recall_score(y_true, y_pred, zero_division=0)),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "roc_auc": float(roc_auc_score(y_true, y_prob)),
            "pr_auc": float(average_precision_score(y_true, y_prob)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
            "brier": float(brier_score_loss(y_true, y_prob)),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        }


def build_baseline_models() -> Dict[str, Any]:
    """Khởi tạo 5 mô hình Baseline chuẩn theo đề cương Tuần 7A."""
    return {
        "Logistic Regression": LogisticRegression(
            class_weight="balanced", C=1.0, max_iter=1000, random_state=42
        ),
        "Decision Tree": DecisionTreeClassifier(
            class_weight="balanced", max_depth=8, min_samples_leaf=25, random_state=42
        ),
        "Random Forest": RandomForestClassifier(
            class_weight="balanced", n_estimators=200, max_depth=12, min_samples_leaf=5, n_jobs=-1, random_state=42
        ),
        "SVM": CalibratedClassifierCV(
            LinearSVC(class_weight="balanced", C=0.5, dual="auto", max_iter=4000, random_state=42),
            cv=3,
            method="sigmoid",
        ),
        "KNN": KNeighborsClassifier(n_neighbors=15, weights="distance", n_jobs=-1),
    }


engine = OULAD5FoldCVEngine()
print(f"Số đặc trưng số học week6_advanced: {len(engine.get_week6_numeric_cols())}")
print(f"Các mô hình Baseline: {list(build_baseline_models().keys())}")

Số đặc trưng số học week6_advanced: 84
Các mô hình Baseline: ['Logistic Regression', 'Decision Tree', 'Random Forest', 'SVM', 'KNN']


## 2. Nạp 4 Lát Cắt Snapshot & Kiểm Chứng Tính Cân Bằng của 5 Fold
Kiểm tra chi tiết sự phân bổ kích thước (~20% mỗi phần), tính phân tầng nhãn `at_risk` và cam kết **$0$ sinh viên trùng lặp** qua từng Fold $1 ightarrow 5$.

In [3]:
snapshot_pcts = [25, 50, 75, 100]
snapshots: Dict[str, pd.DataFrame] = {}
for pct in snapshot_pcts:
    p = SNAPSHOTS_DIR / f"features_snapshot_{pct}pct.csv"
    df_s = pd.read_csv(p)
    snapshots[f"{pct}%"] = df_s
    logger.info(f"Đã nạp {p.name}: {df_s.shape[0]:,} dòng x {df_s.shape[1]} cột")

df_25_dyn = engine.filter_cohort(snapshots["25%"], "dynamic_active")
df_25_full = engine.filter_cohort(snapshots["25%"], "full")

splits_25_dyn, audit_25_dyn = engine.get_5fold_splits(df_25_dyn, n_splits=5, random_state=42)
splits_25_full, audit_25_full = engine.get_5fold_splits(df_25_full, n_splits=5, random_state=42)

print("\n--- KIỂM CHỨNG CHIA 5 PHẦN ĐỀU NHAU: 25% Dynamic Active (N = {:,}) ---".format(len(df_25_dyn)))
display(audit_25_dyn)

print("\n--- KIỂM CHỨNG CHIA 5 PHẦN ĐỀU NHAU: 25% Full Cohort (N = {:,}) ---".format(len(df_25_full)))
display(audit_25_full)

2026-11-19 15:13:25,969 [INFO] Đã nạp features_snapshot_25pct.csv: 32,593 dòng x 113 cột


2026-11-19 15:13:26,434 [INFO] Đã nạp features_snapshot_50pct.csv: 32,593 dòng x 113 cột


2026-11-19 15:13:26,958 [INFO] Đã nạp features_snapshot_75pct.csv: 32,593 dòng x 113 cột


2026-11-19 15:13:27,395 [INFO] Đã nạp features_snapshot_100pct.csv: 32,593 dòng x 113 cột



--- KIỂM CHỨNG CHIA 5 PHẦN ĐỀU NHAU: 25% Dynamic Active (N = 26,248) ---


,fold,n_train,n_test,pct_test,n_train_students,n_test_students,train_at_risk_rate,test_at_risk_rate,student_overlap
0,1,20997,5251,20.0100,19112,4792,0.4138,0.4140,0
1,2,20999,5249,20.0000,19122,4782,0.4139,0.4138,0
2,3,20999,5249,20.0000,19132,4772,0.4139,0.4138,0
3,4,20999,5249,20.0000,19127,4777,0.4139,0.4138,0
4,5,20998,5250,20.0000,19123,4781,0.4138,0.4139,0



--- KIỂM CHỨNG CHIA 5 PHẦN ĐỀU NHAU: 25% Full Cohort (N = 32,593) ---


,fold,n_train,n_test,pct_test,n_train_students,n_test_students,train_at_risk_rate,test_at_risk_rate,student_overlap
0,1,26074,6519,20.0000,23029,5756,0.5280,0.5280,0
1,2,26075,6518,20.0000,23023,5762,0.5280,0.5279,0
2,3,26075,6518,20.0000,23041,5744,0.5280,0.5279,0
3,4,26073,6520,20.0000,23020,5765,0.5279,0.5281,0
4,5,26075,6518,20.0000,23027,5758,0.5280,0.5279,0


## 3. Chứng Minh Định Lượng: Tại Sao 5-Fold CV Giúp "Tránh Ăn May" So Với Chia 1 Lần (Single Hold-Out Split)?
Thử nghiệm chia tập Hold-out 80/20 một lần với 5 seed ngẫu nhiên khác nhau (`seed = 1, 7, 42, 123, 2026`) so với đánh giá đầy đủ **5-Fold Stratified Group CV** tại mốc sớm **25% `dynamic_active`**.

In [4]:
single_split_rows = []
for seed in [1, 7, 42, 123, 2026]:
    s_splits, _ = engine.get_5fold_splits(df_25_dyn, n_splits=5, random_state=seed)
    _, tr_idx, te_idx = s_splits[0]
    X_tr, X_te = engine.prepare_fold_matrices(df_25_dyn.iloc[tr_idx], df_25_dyn.iloc[te_idx])
    y_tr, y_te = df_25_dyn["at_risk"].values[tr_idx], df_25_dyn["at_risk"].values[te_idx]

    for m_name, clf in [
        ("Logistic Regression", build_baseline_models()["Logistic Regression"]),
        ("Random Forest", build_baseline_models()["Random Forest"]),
    ]:
        clf.fit(X_tr, y_tr)
        prob = clf.predict_proba(X_te)[:, 1]
        m = engine.evaluate_predictions(y_te, prob, threshold=0.50)
        single_split_rows.append({
            "strategy": f"Single Split (seed={seed})",
            "seed": seed,
            "model": m_name,
            "n_test": len(te_idx),
            "recall": round(m["recall"], 4),
            "f1": round(m["f1"], 4),
            "roc_auc": round(m["roc_auc"], 4),
            "pr_auc": round(m["pr_auc"], 4),
        })

single_split_df = pd.DataFrame(single_split_rows)
print("--- BIẾN ĐỘNG KHI CHỈ CHIA 1 LẦN (SINGLE HOLD-OUT SPLIT) QUA 5 SEEDS ---")
display(single_split_df)
for m_name in ["Logistic Regression", "Random Forest"]:
    sub = single_split_df[single_split_df["model"] == m_name]
    print(
        f"[{m_name}] Single Split Range -> "
        f"Recall: [{sub['recall'].min():.4f}, {sub['recall'].max():.4f}] (chênh {(sub['recall'].max()-sub['recall'].min())*100:.2f}%), "
        f"AUC-ROC: [{sub['roc_auc'].min():.4f}, {sub['roc_auc'].max():.4f}]"
    )

--- BIẾN ĐỘNG KHI CHỈ CHIA 1 LẦN (SINGLE HOLD-OUT SPLIT) QUA 5 SEEDS ---


,strategy,seed,model,n_test,recall,f1,roc_auc,pr_auc
0,Single Split (seed=1),1,Logistic Regression,5249,0.7247,0.7279,0.8478,0.8263
1,Single Split (seed=1),1,Random Forest,5249,0.6892,0.7214,0.8484,0.8268
2,Single Split (seed=7),7,Logistic Regression,5251,0.7061,0.7185,0.8420,0.8202
3,Single Split (seed=7),7,Random Forest,5251,0.6665,0.7075,0.8436,0.8229
4,Single Split (seed=42),42,Logistic Regression,5251,0.6914,0.7075,0.8395,0.8151
5,Single Split (seed=42),42,Random Forest,5251,0.6559,0.7004,0.8419,0.8179
6,Single Split (seed=123),123,Logistic Regression,5251,0.6895,0.7079,0.8388,0.8128
7,Single Split (seed=123),123,Random Forest,5251,0.6610,0.7081,0.8408,0.8143
8,Single Split (seed=2026),2026,Logistic Regression,5249,0.7164,0.7219,0.8448,0.8252
9,Single Split (seed=2026),2026,Random Forest,5249,0.6796,0.7108,0.8438,0.8245


[Logistic Regression] Single Split Range -> Recall: [0.6895, 0.7247] (chênh 3.52%), AUC-ROC: [0.8388, 0.8478]
[Random Forest] Single Split Range -> Recall: [0.6559, 0.6892] (chênh 3.33%), AUC-ROC: [0.8408, 0.8484]


## 4. Thực Nghiệm Đánh Giá 5 Mô Hình Baseline Qua 5-Fold Stratified Group CV
Chạy đầy đủ cả **5 Fold (Fold 1 $ightarrow$ Fold 5)** cho toàn bộ 5 mô hình Baseline (`Logistic Regression`, `Decision Tree`, `Random Forest`, `SVM`, `KNN`) trên 6 cấu hình chuẩn:
- Cohort `dynamic_active` tại 4 mốc: `25%`, `50%`, `75%`, `100%`
- Cohort `full` tại 2 mốc: `25%`, `100%`

In [5]:
experiment_configs = [
    ("25%", "dynamic_active"),
    ("50%", "dynamic_active"),
    ("75%", "dynamic_active"),
    ("100%", "dynamic_active"),
    ("25%", "full"),
    ("100%", "full"),
]

per_fold_records: List[Dict[str, Any]] = []
summary_records: List[Dict[str, Any]] = []
oof_store: Dict[str, Dict[str, Any]] = {}
fold_audit_map: Dict[str, List[Dict[str, Any]]] = {}

metric_keys = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "balanced_accuracy", "brier"]

for snap_label, cohort_name in experiment_configs:
    t0_cfg = time.time()
    df_c = engine.filter_cohort(snapshots[snap_label], cohort_name)
    y_all = df_c["at_risk"].values
    splits, audit_df = engine.get_5fold_splits(df_c, n_splits=5, random_state=42)
    cfg_key = f"{snap_label}_{cohort_name}"
    fold_audit_map[cfg_key] = audit_df.to_dict(orient="records")

    model_names = list(build_baseline_models().keys())
    oof_probs: Dict[str, np.ndarray] = {m: np.zeros(len(df_c), dtype=float) for m in model_names}
    fold_ids = np.zeros(len(df_c), dtype=int)
    model_fold_metrics: Dict[str, List[Dict[str, Any]]] = {m: [] for m in model_names}

    for fold, tr_idx, te_idx in splits:
        fold_ids[te_idx] = fold
        X_tr, X_te = engine.prepare_fold_matrices(df_c.iloc[tr_idx], df_c.iloc[te_idx])
        y_tr, y_te = y_all[tr_idx], y_all[te_idx]

        builders = build_baseline_models()
        for m_name, clf in builders.items():
            t_start = time.time()
            clf.fit(X_tr, y_tr)
            prob_te = clf.predict_proba(X_te)[:, 1]
            fit_sec = time.time() - t_start

            oof_probs[m_name][te_idx] = prob_te
            m_res = engine.evaluate_predictions(y_te, prob_te, threshold=0.50)
            row = {
                "snapshot": snap_label,
                "cohort": cohort_name,
                "n_cohort": int(len(df_c)),
                "model": m_name,
                "fold": int(fold),
                "n_train": int(len(tr_idx)),
                "n_test": int(len(te_idx)),
                "fit_seconds": round(fit_sec, 3),
                **{k: round(m_res[k], 4) for k in metric_keys},
                "tn": m_res["tn"], "fp": m_res["fp"], "fn": m_res["fn"], "tp": m_res["tp"],
            }
            per_fold_records.append(row)
            model_fold_metrics[m_name].append(row)

    for m_name in model_names:
        f_df = pd.DataFrame(model_fold_metrics[m_name])
        oof_m = engine.evaluate_predictions(y_all, oof_probs[m_name], threshold=0.50)
        sum_row: Dict[str, Any] = {
            "snapshot": snap_label,
            "cohort": cohort_name,
            "n_cohort": int(len(df_c)),
            "at_risk_rate": round(float(y_all.mean()), 4),
            "model": m_name,
            "mean_fit_seconds": round(float(f_df["fit_seconds"].mean()), 3),
        }
        for k in metric_keys:
            sum_row[f"{k}_mean"] = round(float(f_df[k].mean()), 4)
            sum_row[f"{k}_sd"] = round(float(f_df[k].std(ddof=1)), 4)
            sum_row[f"{k}_oof"] = round(float(oof_m[k]), 4)
        sum_row["oof_tn"] = oof_m["tn"]
        sum_row["oof_fp"] = oof_m["fp"]
        sum_row["oof_fn"] = oof_m["fn"]
        sum_row["oof_tp"] = oof_m["tp"]
        summary_records.append(sum_row)

    if snap_label in ["25%", "100%"] and cohort_name == "dynamic_active":
        oof_store[cfg_key] = {
            "id_student": df_c["id_student"].astype(int).tolist(),
            "code_module": df_c["code_module"].astype(str).tolist(),
            "code_presentation": df_c["code_presentation"].astype(str).tolist(),
            "final_result": df_c["final_result"].astype(str).tolist(),
            "y_true": y_all.astype(int).tolist(),
            "fold_ids": fold_ids.astype(int).tolist(),
            "probs": {m: np.round(oof_probs[m], 6).tolist() for m in model_names},
        }

    logger.info(f"Hoàn tất 5-Fold CV cho ({snap_label}, {cohort_name}) trong {time.time() - t0_cfg:.1f}s")

per_fold_df = pd.DataFrame(per_fold_records)
summary_df = pd.DataFrame(summary_records)

print("\n=== BẢNG TỔNG HỢP 5-FOLD CV BASELINES TẠI MỐC SỚM 25% (DYNAMIC ACTIVE) ===")
disp_25 = summary_df[(summary_df["snapshot"] == "25%") & (summary_df["cohort"] == "dynamic_active")].sort_values("roc_auc_mean", ascending=False)
display(disp_25[["model", "accuracy_mean", "accuracy_sd", "precision_mean", "recall_mean", "recall_sd", "f1_mean", "f1_sd", "roc_auc_mean", "roc_auc_sd", "pr_auc_mean", "brier_mean"]])

print("\n=== CHI TIẾT TỪNG FOLD (FOLD 1 -> FOLD 5) TẠI MỐC 25% DYNAMIC ACTIVE ===")
f25_pivot = per_fold_df[(per_fold_df["snapshot"] == "25%") & (per_fold_df["cohort"] == "dynamic_active")]
display(f25_pivot[["model", "fold", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]])

2026-11-19 15:15:36,261 [INFO] Hoàn tất 5-Fold CV cho (25%, dynamic_active) trong 64.2s


2026-11-19 15:16:27,820 [INFO] Hoàn tất 5-Fold CV cho (50%, dynamic_active) trong 51.6s


2026-11-19 15:17:15,912 [INFO] Hoàn tất 5-Fold CV cho (75%, dynamic_active) trong 48.1s


2026-11-19 15:18:04,112 [INFO] Hoàn tất 5-Fold CV cho (100%, dynamic_active) trong 48.2s


2026-11-19 15:19:11,978 [INFO] Hoàn tất 5-Fold CV cho (25%, full) trong 67.9s


2026-11-19 15:20:10,684 [INFO] Hoàn tất 5-Fold CV cho (100%, full) trong 58.7s



=== BẢNG TỔNG HỢP 5-FOLD CV BASELINES TẠI MỐC SỚM 25% (DYNAMIC ACTIVE) ===


,model,accuracy_mean,accuracy_sd,precision_mean,recall_mean,recall_sd,f1_mean,f1_sd,roc_auc_mean,roc_auc_sd,pr_auc_mean,brier_mean
0,Logistic Regression,0.7687,0.0058,0.7254,0.7099,0.0112,0.7175,0.0067,0.8423,0.0027,0.8212,0.1577
2,Random Forest,0.7733,0.0058,0.7524,0.6745,0.0105,0.7113,0.0073,0.8416,0.0040,0.8204,0.1569
3,SVM,0.7459,0.0076,0.8772,0.4491,0.0221,0.5937,0.0186,0.8206,0.0074,0.7989,0.1799
1,Decision Tree,0.7373,0.0097,0.6897,0.6694,0.0398,0.6781,0.0102,0.8093,0.0063,0.7783,0.1740
4,KNN,0.7534,0.0033,0.8050,0.5334,0.0121,0.6415,0.0081,0.8071,0.0048,0.7865,0.1707



=== CHI TIẾT TỪNG FOLD (FOLD 1 -> FOLD 5) TẠI MỐC 25% DYNAMIC ACTIVE ===


,model,fold,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Logistic Regression,1,0.7633,0.7243,0.6914,0.7075,0.8395,0.8151
1,Decision Tree,1,0.7456,0.7280,0.6155,0.6670,0.8075,0.7723
2,Random Forest,1,0.7677,0.7513,0.6559,0.7004,0.8419,0.8179
3,SVM,1,0.7368,0.8694,0.4287,0.5742,0.8139,0.7903
4,KNN,1,0.7494,0.8109,0.5147,0.6297,0.8017,0.7814
5,Logistic Regression,2,0.7714,0.7256,0.7196,0.7226,0.8403,0.8203
6,Decision Tree,2,0.7295,0.6799,0.6542,0.6668,0.7998,0.7695
7,Random Forest,2,0.7723,0.7468,0.6805,0.7121,0.8362,0.8153
8,SVM,2,0.7550,0.8819,0.4710,0.6140,0.8311,0.8111
9,KNN,2,0.7558,0.8035,0.5424,0.6476,0.8064,0.7841


## 5. Phân Tích Lỗi OOF (Error Analysis) & Chọn Top 3 Mô Hình Baseline Tốt Nhất
Khảo sát chi tiết cơ cấu dự đoán sai (`FN` - bỏ sót sinh viên nguy cơ và `FP` - cảnh báo nhầm) trên toàn bộ $26,248$ sinh viên tại mốc sớm **25% `dynamic_active`**.

In [6]:
oof_25_meta = oof_store["25%_dynamic_active"]
y_true_25 = np.array(oof_25_meta["y_true"])
res_25 = np.array(oof_25_meta["final_result"])
df_25_err = df_25_dyn.copy()

top3_baselines = (
    summary_df[(summary_df["snapshot"] == "25%") & (summary_df["cohort"] == "dynamic_active")]
    .sort_values(["roc_auc_mean", "f1_mean"], ascending=False)["model"]
    .head(3)
    .tolist()
)
print(f"Top 3 mô hình Baseline được chọn chuyển sang Giai đoạn 7B: {top3_baselines}")

error_analysis_summary = {}
for m_name in ["Random Forest", "Logistic Regression", "SVM"]:
    prob_m = np.array(oof_25_meta["probs"][m_name])
    pred_m = (prob_m >= 0.50).astype(int)
    fn_mask = (y_true_25 == 1) & (pred_m == 0)
    fp_mask = (y_true_25 == 0) & (pred_m == 1)
    tp_mask = (y_true_25 == 1) & (pred_m == 1)
    tn_mask = (y_true_25 == 0) & (pred_m == 0)

    fn_outcomes = pd.Series(res_25[fn_mask]).value_counts().to_dict()
    fp_outcomes = pd.Series(res_25[fp_mask]).value_counts().to_dict()

    profile_cols = ["total_clicks", "active_days", "submission_rate", "days_since_last_activity", "engagement_decay"]
    group_profiles = {}
    for g_label, g_mask in [("TP", tp_mask), ("FN", fn_mask), ("TN", tn_mask), ("FP", fp_mask)]:
        sub_g = df_25_err.loc[g_mask, profile_cols].mean()
        group_profiles[g_label] = {c: round(float(sub_g[c]), 3) for c in profile_cols}

    error_analysis_summary[m_name] = {
        "n_tp": int(tp_mask.sum()),
        "n_fn": int(fn_mask.sum()),
        "n_tn": int(tn_mask.sum()),
        "n_fp": int(fp_mask.sum()),
        "fn_final_result_breakdown": {str(k): int(v) for k, v in fn_outcomes.items()},
        "fp_final_result_breakdown": {str(k): int(v) for k, v in fp_outcomes.items()},
        "feature_means_by_outcome": group_profiles,
    }

print("\n--- PHÂN TÍCH LỖI CHI TIẾT CHO RANDOM FOREST OOF (MỐC 25% DYNAMIC ACTIVE) ---")
rf_err = error_analysis_summary["Random Forest"]
print(f"TP={rf_err['n_tp']:,}, FN={rf_err['n_fn']:,} (bỏ sót: {rf_err['fn_final_result_breakdown']})")
print(f"TN={rf_err['n_tn']:,}, FP={rf_err['n_fp']:,} (cảnh báo nhầm: {rf_err['fp_final_result_breakdown']})")
display(pd.DataFrame(rf_err["feature_means_by_outcome"]).T)

Top 3 mô hình Baseline được chọn chuyển sang Giai đoạn 7B: ['Logistic Regression', 'Random Forest', 'SVM']

--- PHÂN TÍCH LỖI CHI TIẾT CHO RANDOM FOREST OOF (MỐC 25% DYNAMIC ACTIVE) ---
TP=7,327, FN=3,536 (bỏ sót: {'Fail': 2139, 'Withdrawn': 1397})
TN=12,972, FP=2,413 (cảnh báo nhầm: {'Pass': 2327, 'Distinction': 86})


,total_clicks,active_days,submission_rate,days_since_last_activity,engagement_decay
TP,274.3400,14.6950,0.5790,20.2840,0.8330
FN,654.0390,29.3100,0.9290,4.1850,0.8320
TN,829.3950,34.4180,0.9430,3.1000,0.5950
FP,358.8280,19.2510,0.8300,8.6190,1.6010


## 6. Trực Quan Hóa 5-Fold CV & Lưu Kết Quả Baseline (Hình 31–33)
Xuất 3 biểu đồ chất lượng cao (300 DPI) tại `reports/figures/` và lưu kết quả đối chuẩn vào `data/processed/week7_modeling/week7_baseline_cv_metrics.json`.

In [7]:
# =========================================================================
# HÌNH 31: 31_week7_5fold_cv_architecture_and_baseline_comparison.png
# =========================================================================
logger.info("Đang vẽ Hình 31: 31_week7_5fold_cv_architecture_and_baseline_comparison.png")
fig, axes = plt.subplots(1, 3, figsize=(19.5, 6.2))

# Panel A: Phân bổ 5 Fold đều nhau & Tỷ lệ At-Risk
ax = axes[0]
folds = audit_25_dyn["fold"].values
n_tests = audit_25_dyn["n_test"].values
ar_rates = audit_25_dyn["test_at_risk_rate"].values * 100.0
bars = ax.bar(folds, n_tests, color="#3498db", edgecolor="black", alpha=0.85, width=0.58, label="Số SV Test mỗi Fold (~20%)")
ax.set_title("(A) Kiểm chứng Chia 5 Phần Đều Nhau (0% Trùng SV)", fontsize=11.5, fontweight="bold")
ax.set_xlabel("Fold kiểm thử (Test Fold k = 1..5)", fontsize=10.5, fontweight="bold")
ax.set_ylabel("Số bản ghi Test (Dynamic Active 25%)", fontsize=10.5, fontweight="bold")
ax.set_xticks(folds)
ax.set_xticklabels([f"Fold {f}" for f in folds])
ax.set_ylim(0, max(n_tests) * 1.25)
for b, cnt, pct, ar in zip(bars, n_tests, audit_25_dyn["pct_test"].values, ar_rates):
    ax.text(
        b.get_x() + b.get_width() / 2.0,
        cnt + 140,
        f"{cnt:,}\n({pct:.1f}%)\nAR: {ar:.1f}%",
        ha="center",
        va="bottom",
        fontsize=8.3,
        fontweight="bold",
    )
ax.grid(axis="y", alpha=0.3)

# Panel B: 5-Baseline Comparison Mean +- SD (25% Dynamic Active)
ax = axes[1]
sub_25 = summary_df[(summary_df["snapshot"] == "25%") & (summary_df["cohort"] == "dynamic_active")].sort_values("roc_auc_mean", ascending=True)
y_pos = np.arange(len(sub_25))
h = 0.24
ax.barh(y_pos + h, sub_25["roc_auc_mean"], xerr=sub_25["roc_auc_sd"], height=h, color="#2980b9", edgecolor="black", label="AUC-ROC", capsize=3)
ax.barh(y_pos, sub_25["f1_mean"], xerr=sub_25["f1_sd"], height=h, color="#27ae60", edgecolor="black", label="F1-Score", capsize=3)
ax.barh(y_pos - h, sub_25["recall_mean"], xerr=sub_25["recall_sd"], height=h, color="#e67e22", edgecolor="black", label="Recall (At-Risk)", capsize=3)
ax.axvline(0.75, color="#c0392b", linestyle="--", linewidth=1.4, label="Mục tiêu Recall = 0.75")
ax.set_yticks(y_pos)
ax.set_yticklabels(sub_25["model"], fontsize=9.5, fontweight="bold")
ax.set_xlim(0.40, 0.96)
ax.set_xlabel("Điểm số Trung bình 5-Fold (Mean ± SD)", fontsize=10.5, fontweight="bold")
ax.set_title("(B) So sánh 5 Mô hình Baseline (25% Dynamic Active)", fontsize=11.5, fontweight="bold")
for yp, auc_v, f1_v, rec_v in zip(y_pos, sub_25["roc_auc_mean"], sub_25["f1_mean"], sub_25["recall_mean"]):
    ax.text(auc_v + 0.008, yp + h, f"{auc_v:.3f}", va="center", fontsize=7.8, fontweight="bold", color="#1f618d")
    ax.text(f1_v + 0.008, yp, f"{f1_v:.3f}", va="center", fontsize=7.8, fontweight="bold", color="#1e8449")
    ax.text(rec_v + 0.008, yp - h, f"{rec_v:.3f}", va="center", fontsize=7.8, fontweight="bold", color="#a04000")
ax.legend(loc="lower right", fontsize=8.3)
ax.grid(axis="x", alpha=0.3)

# Panel C: Độ ổn định AUC-ROC qua từng Fold 1..5
ax = axes[2]
m_palette = {
    "Random Forest": ("#27ae60", "o"),
    "Logistic Regression": ("#2980b9", "s"),
    "SVM": ("#8e44ad", "^"),
    "Decision Tree": ("#e67e22", "D"),
    "KNN": ("#7f8c8d", "v"),
}
for m_name, (col, mk) in m_palette.items():
    sub_f = f25_pivot[f25_pivot["model"] == m_name].sort_values("fold")
    ax.plot(sub_f["fold"], sub_f["roc_auc"], color=col, marker=mk, linewidth=2.1, markersize=6.5, label=m_name)
ax.axhline(0.80, color="#c0392b", linestyle=":", linewidth=1.5, label="Ngưỡng AUC = 0.80")
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels([f"Fold {i}" for i in range(1, 6)])
ax.set_ylim(0.72, 0.89)
ax.set_xlabel("Lần kiểm thử (Test Fold)", fontsize=10.5, fontweight="bold")
ax.set_ylabel("Test AUC-ROC từng Fold", fontsize=10.5, fontweight="bold")
ax.set_title("(C) Độ ổn định Hiệu năng qua Từng Fold (1 -> 5)", fontsize=11.5, fontweight="bold")
ax.legend(loc="lower right", fontsize=8.2)
ax.grid(alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "31_week7_5fold_cv_architecture_and_baseline_comparison.png", dpi=300)
plt.close(fig)

# =========================================================================
# HÌNH 32: 32_week7_baseline_oof_confusion_matrices.png
# =========================================================================
logger.info("Đang vẽ Hình 32: 32_week7_baseline_oof_confusion_matrices.png")
fig, axes = plt.subplots(2, 3, figsize=(17.5, 10.5))
axes_flat = axes.ravel()
ordered_models = ["Random Forest", "Logistic Regression", "SVM", "Decision Tree", "KNN"]

for idx, m_name in enumerate(ordered_models):
    ax = axes_flat[idx]
    prob = np.array(oof_25_meta["probs"][m_name])
    pred = (prob >= 0.50).astype(int)
    cm = confusion_matrix(y_true_25, pred, labels=[0, 1])
    cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100.0
    annot = np.array([
        [f"{cm[r, c]:,}\n({cm_pct[r, c]:.1f}%)" for c in range(2)]
        for r in range(2)
    ])
    sns.heatmap(
        cm_pct,
        annot=annot,
        fmt="",
        cmap="Blues",
        cbar=False,
        vmin=0,
        vmax=100,
        linewidths=1.0,
        linecolor="black",
        annot_kws={"size": 10.5, "weight": "bold"},
        ax=ax,
    )
    rec_v = recall_score(y_true_25, pred)
    f1_v = f1_score(y_true_25, pred)
    auc_v = roc_auc_score(y_true_25, prob)
    ax.set_title(f"{m_name}\nOOF Recall={rec_v:.3f} | F1={f1_v:.3f} | AUC={auc_v:.3f}", fontsize=10.8, fontweight="bold")
    ax.set_xticklabels(["Dự đoán An toàn (0)", "Dự đoán At-Risk (1)"], fontsize=9)
    ax.set_yticklabels(["Thực tế An toàn (0)", "Thực tế At-Risk (1)"], fontsize=9, rotation=0)

# Panel 6: So sánh Số lượng FN (Bỏ sót) vs FP (Cảnh báo nhầm) trên OOF
ax_err = axes_flat[5]
err_rows = []
for m_name in ordered_models:
    prob = np.array(oof_25_meta["probs"][m_name])
    pred = (prob >= 0.50).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_25, pred, labels=[0, 1]).ravel()
    err_rows.append({"model": m_name, "FN (Bỏ sót At-Risk)": int(fn), "FP (Cảnh báo nhầm)": int(fp)})
err_df = pd.DataFrame(err_rows)
x_e = np.arange(len(err_df))
ax_err.bar(x_e - 0.18, err_df["FN (Bỏ sót At-Risk)"], width=0.35, color="#c0392b", edgecolor="black", label="FN (Bỏ sót At-Risk)")
ax_err.bar(x_e + 0.18, err_df["FP (Cảnh báo nhầm)"], width=0.35, color="#f39c12", edgecolor="black", label="FP (Cảnh báo nhầm)")
ax_err.set_xticks(x_e)
ax_err.set_xticklabels(["RF", "LogReg", "SVM", "DT", "KNN"], fontsize=9.5, fontweight="bold")
ax_err.set_ylabel("Số lượt sinh viên OOF (N = 26,248)", fontsize=10, fontweight="bold")
ax_err.set_title("Tổng hợp Lỗi OOF (FN vs FP) tại ngưỡng tau = 0.50", fontsize=10.8, fontweight="bold")
for i, (fn_v, fp_v) in enumerate(zip(err_df["FN (Bỏ sót At-Risk)"], err_df["FP (Cảnh báo nhầm)"])):
    ax_err.text(i - 0.18, fn_v + 80, f"{fn_v:,}", ha="center", fontsize=8.2, fontweight="bold", color="#922b21")
    ax_err.text(i + 0.18, fp_v + 80, f"{fp_v:,}", ha="center", fontsize=8.2, fontweight="bold", color="#9c640c")
ax_err.set_ylim(0, max(err_df["FN (Bỏ sót At-Risk)"].max(), err_df["FP (Cảnh báo nhầm)"].max()) * 1.18)
ax_err.legend(loc="upper left", fontsize=8.8)
ax_err.grid(axis="y", alpha=0.3)

plt.suptitle("Ma trận Nhầm lẫn Out-of-Fold (100% Sinh viên, 5-Fold CV) — Mốc 25% Dynamic Active", fontsize=13.5, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "32_week7_baseline_oof_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# =========================================================================
# HÌNH 33: 33_week7_baseline_roc_and_pr_curves_5fold.png
# =========================================================================
logger.info("Đang vẽ Hình 33: 33_week7_baseline_roc_and_pr_curves_5fold.png")
fig, axes = plt.subplots(1, 2, figsize=(16.5, 6.5))

for m_name, (col, _) in m_palette.items():
    prob = np.array(oof_25_meta["probs"][m_name])
    fpr, tpr, _ = roc_curve(y_true_25, prob)
    auc_v = roc_auc_score(y_true_25, prob)
    axes[0].plot(fpr, tpr, color=col, linewidth=2.2, label=f"{m_name} (OOF AUC = {auc_v:.4f})")

    prec_c, rec_c, _ = precision_recall_curve(y_true_25, prob)
    ap_v = average_precision_score(y_true_25, prob)
    axes[1].plot(rec_c, prec_c, color=col, linewidth=2.2, label=f"{m_name} (OOF PR-AUC = {ap_v:.4f})")

axes[0].plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Random Baseline (AUC = 0.50)")
axes[0].set_title("(A) Đường cong ROC Out-of-Fold (5-Fold CV, Mốc 25% Dynamic)", fontsize=11.8, fontweight="bold")
axes[0].set_xlabel("False Positive Rate (1 - Specificity)", fontsize=10.5, fontweight="bold")
axes[0].set_ylabel("True Positive Rate (Recall)", fontsize=10.5, fontweight="bold")
axes[0].legend(loc="lower right", fontsize=9.0)
axes[0].grid(alpha=0.3)

prev = float(y_true_25.mean())
axes[1].axhline(prev, color="black", linestyle="--", linewidth=1.2, label=f"No-Skill Baseline (At-Risk = {prev:.3f})")
axes[1].set_title("(B) Đường cong Precision-Recall OOF (5-Fold CV, Mốc 25% Dynamic)", fontsize=11.8, fontweight="bold")
axes[1].set_xlabel("Recall (Độ nhạy phát hiện sinh viên At-Risk)", fontsize=10.5, fontweight="bold")
axes[1].set_ylabel("Precision (Độ chính xác cảnh báo)", fontsize=10.5, fontweight="bold")
axes[1].legend(loc="lower left", fontsize=9.0)
axes[1].grid(alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "33_week7_baseline_roc_and_pr_curves_5fold.png", dpi=300)
plt.close(fig)

# Lưu kết quả Baseline ra data/processed/week7_modeling/week7_baseline_cv_metrics.json
baseline_payload = {
    "protocol": {
        "cv_strategy": "StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)",
        "group_col": "id_student",
        "feature_set": "week6_advanced",
        "n_features_encoded": 114,
        "zero_data_leakage": "In-fold fitting of Imputation, IQR Capping, Log1p, OneHotEncoder, RobustScaler",
    },
    "fold_audit": fold_audit_map,
    "single_split_vs_5fold": single_split_rows,
    "per_fold_results": per_fold_records,
    "summary_results": summary_records,
    "top3_selected_baselines": top3_baselines,
    "error_analysis_25_dyn": error_analysis_summary,
    "oof_predictions": oof_store,
}
baseline_out_path = WEEK7_PROCESSED_DIR / "week7_baseline_cv_metrics.json"
with open(baseline_out_path, "w", encoding="utf-8") as f:
    json.dump(baseline_payload, f, indent=2, ensure_ascii=False)
logger.info(f"Đã lưu kết quả Baseline 5-Fold CV tại: {baseline_out_path} ({baseline_out_path.stat().st_size / 1e6:.2f} MB)")

2026-11-19 15:20:10,801 [INFO] Đang vẽ Hình 31: 31_week7_5fold_cv_architecture_and_baseline_comparison.png


2026-11-19 15:20:12,546 [INFO] Đang vẽ Hình 32: 32_week7_baseline_oof_confusion_matrices.png


2026-11-19 15:20:14,758 [INFO] Đang vẽ Hình 33: 33_week7_baseline_roc_and_pr_curves_5fold.png


2026-11-19 15:20:16,296 [INFO] Đã lưu kết quả Baseline 5-Fold CV tại: C:\Users\mrore\Project_I\data\processed\week7_baseline_cv_metrics.json (9.67 MB)


## 7. Bộ Kiểm Thử Tự Động `TestWeek7BaselineModels` (8/8 Unit Tests)
Xác minh độc lập tính đúng đắn của phân vùng 5 phần, không rò rỉ sinh viên hay thông số tiền xử lý giữa các Fold, đủ 5 mô hình Baseline và sự tồn tại của mọi artifacts đầu ra.

In [8]:
class TestWeek7BaselineModels(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.engine = OULAD5FoldCVEngine()
        cls.df_25 = pd.read_csv(SNAPSHOTS_DIR / "features_snapshot_25pct.csv")
        cls.df_25_dyn = cls.engine.filter_cohort(cls.df_25, "dynamic_active")
        with open(WEEK7_PROCESSED_DIR / "week7_baseline_cv_metrics.json", "r", encoding="utf-8") as f:
            cls.payload = json.load(f)

    def test_01_snapshots_loaded_and_valid_shape(self):
        self.assertEqual(len(self.df_25), 32593)
        self.assertGreater(len(self.df_25_dyn), 25000)

    def test_02_5fold_equal_partitions_and_stratification(self):
        splits, audit = self.engine.get_5fold_splits(self.df_25_dyn, n_splits=5, random_state=42)
        self.assertEqual(len(splits), 5)
        self.assertTrue((audit["pct_test"] >= 18.5).all() and (audit["pct_test"] <= 21.5).all())
        rate_spread = audit["test_at_risk_rate"].max() - audit["test_at_risk_rate"].min()
        self.assertLess(rate_spread, 0.02)

    def test_03_zero_student_overlap_across_all_5_folds(self):
        for cfg_key, rows in self.payload["fold_audit"].items():
            self.assertEqual(len(rows), 5, f"Thiếu fold ở {cfg_key}")
            for r in rows:
                self.assertEqual(r["student_overlap"], 0, f"Phát hiện rò rỉ sinh viên tại {cfg_key} fold {r['fold']}")

    def test_04_infold_preprocessing_zero_data_leakage(self):
        splits, _ = self.engine.get_5fold_splits(self.df_25_dyn, n_splits=5, random_state=42)
        _, tr_idx, te_idx = splits[0]
        tr_df = self.df_25_dyn.iloc[tr_idx].copy()
        te_orig = self.df_25_dyn.iloc[te_idx].copy()
        te_mut = te_orig.copy()
        te_mut["total_clicks"] = 999999.0
        te_mut["avg_score"] = 100.0
        te_mut["imd_band"] = "0-10%"

        X_tr1, _ = self.engine.prepare_fold_matrices(tr_df, te_orig)
        X_tr2, X_te2 = self.engine.prepare_fold_matrices(tr_df, te_mut)
        self.assertEqual(X_tr1.shape[1], 114)
        self.assertTrue(np.allclose(X_tr1.values, X_tr2.values, atol=1e-12))
        self.assertFalse(np.isnan(X_te2.values).any())
        self.assertFalse(np.isinf(X_te2.values).any())

    def test_05_all_5_baselines_evaluated_across_5_folds(self):
        pf = pd.DataFrame(self.payload["per_fold_results"])
        sub = pf[(pf["snapshot"] == "25%") & (pf["cohort"] == "dynamic_active")]
        expected_models = {"Logistic Regression", "Decision Tree", "Random Forest", "SVM", "KNN"}
        self.assertEqual(set(sub["model"].unique()), expected_models)
        for m in expected_models:
            self.assertEqual(sorted(sub[sub["model"] == m]["fold"].tolist()), [1, 2, 3, 4, 5])

    def test_06_oof_predictions_cover_100_percent_samples(self):
        oof = self.payload["oof_predictions"]["25%_dynamic_active"]
        n = len(self.df_25_dyn)
        self.assertEqual(len(oof["y_true"]), n)
        self.assertEqual(len(oof["fold_ids"]), n)
        self.assertEqual(set(oof["fold_ids"]), {1, 2, 3, 4, 5})
        for m_name, probs in oof["probs"].items():
            self.assertEqual(len(probs), n, f"OOF thiếu mẫu ở {m_name}")

    def test_07_metrics_consistency_and_valid_ranges(self):
        sm = pd.DataFrame(self.payload["summary_results"])
        for col in ["accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "roc_auc_mean", "pr_auc_mean"]:
            self.assertTrue((sm[col] >= 0.0).all() and (sm[col] <= 1.0).all(), f"Giá trị ngoài [0,1] ở {col}")
        sub_25 = sm[(sm["snapshot"] == "25%") & (sm["cohort"] == "dynamic_active")]
        self.assertGreaterEqual(sub_25["roc_auc_mean"].max(), 0.82)

    def test_08_artifacts_and_figures_saved(self):
        self.assertTrue((WEEK7_PROCESSED_DIR / "week7_baseline_cv_metrics.json").exists())
        self.assertFalse(any(p.is_file() for p in PROCESSED_DIR.iterdir()), "Còn tệp rời tại gốc data/processed/!")
        for fig_name in [
            "31_week7_5fold_cv_architecture_and_baseline_comparison.png",
            "32_week7_baseline_oof_confusion_matrices.png",
            "33_week7_baseline_roc_and_pr_curves_5fold.png",
        ]:
            p = FIGURES_DIR / fig_name
            self.assertTrue(p.exists(), f"Thiếu biểu đồ: {fig_name}")
            self.assertGreater(p.stat().st_size, 10000)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestWeek7BaselineModels)
result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
assert result.wasSuccessful(), f"Có test thất bại: failures={len(result.failures)}, errors={len(result.errors)}"
print(f"\n[PASSED] {result.testsRun}/{result.testsRun} kiểm thử Giai đoạn 7A (05_baseline_models.ipynb) hoàn tất không lỗi!")

test_01_snapshots_loaded_and_valid_shape (__main__.TestWeek7BaselineModels.test_01_snapshots_loaded_and_valid_shape) ... 

ok


test_02_5fold_equal_partitions_and_stratification (__main__.TestWeek7BaselineModels.test_02_5fold_equal_partitions_and_stratification) ... 

ok


test_03_zero_student_overlap_across_all_5_folds (__main__.TestWeek7BaselineModels.test_03_zero_student_overlap_across_all_5_folds) ... 

ok


... (8 outputs left out)

test_08_artifacts_and_figures_saved (__main__.TestWeek7BaselineModels.test_08_artifacts_and_figures_saved) ... 

ok


----------------------------------------------------------------------
Ran 8 tests in 12.366s

OK



[PASSED] 8/8 kiểm thử Giai đoạn 7A (05_baseline_models.ipynb) hoàn tất không lỗi!
